In [1]:
from dotenv import load_dotenv
import os
from google import genai
from google.genai import types

genai.__version__

'1.7.0'

In [2]:
#Importing google api key
load_dotenv()
api_key = {}

api_lst = ['GOOGLE_API_KEY']

for key in api_lst:
    value = os.environ.get(key)
    if value is not None:
        api_key[key] = value
    else:
        print(f"{key} does not exist in environment")

googl_api_key = api_key['GOOGLE_API_KEY']

In [3]:
# Define a retry policy. The model might make multiple consecutive calls automatically
# for a complex query, this ensures the client retries if it hits quota limits.
from google.api_core import retry

is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

if not hasattr(genai.models.Models.generate_content, '__wrapped__'):
  genai.models.Models.generate_content = retry.Retry(
      predicate=is_retriable)(genai.models.Models.generate_content)

In [4]:
import sqlite3
import os

db_name = "identifier.sqlite.db"
if not os.path.exists(db_name):
    print(f"Database file {db_name} does not exist")
else:
    conn = sqlite3.connect(db_name)

In [5]:
cursor = conn.cursor()
#Creating tables
cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
product_id INTEGER PRIMARY KEY AUTOINCREMENT,
product_name VARCHAR(255) NOT NULL,
price DECIMAL(10, 2) NOT NULL);
""")

cursor.execute(""" CREATE TABLE IF NOT EXISTS staff (
staff_id INTEGER PRIMARY KEY AUTOINCREMENT,
first_name VARCHAR(255) NOT NULL,
last_name VARCHAR(255) NOT NULL);""")

cursor.execute(""" CREATE TABLE IF NOT EXISTS orders (
order_id INTEGER PRIMARY KEY AUTOINCREMENT,
customer_name VARCHAR(255) NOT NULL,
staff_id INTEGER NOT NULL,
product_id INTEGER NOT NULL,
FOREIGN KEY (staff_id) REFERENCES staff (staff_id),
FOREIGN KEY (product_id) REFERENCES products (product_id));
""")

cursor.execute("""
INSERT INTO products (product_name, price) VALUES
('Laptop', 799.99),
('Keyboard', 129.99),
('Mouse', 29.99);
""")

cursor.execute("""INSERT INTO staff (first_name, last_name) VALUES
('Alice', 'Smith'),
('Bob', 'Johnson'),
('Charlie', 'Williams');""")

cursor.execute("""INSERT INTO orders (customer_name, staff_id, product_id) VALUES
('David Lee', 1, 1),
('Emily Chen', 2, 2),
('Frank Brown', 1, 3);
""")

In [6]:
#connecting to db to confirm if table exist
def table_list():

    cursor = conn.cursor()
    cursor.execute("""SELECT name FROM sqlite_master WHERE type='table';""")
    tables = cursor.fetchall()

    return [t[0] for t in tables]

table_list()

['productions', 'sqlite_sequence', 'staff', 'orders', 'products']

In [7]:
def describe_table(table_name: str) -> list[tuple[str, str]]:
    """Look up the table schema.

    Returns:
      List of columns, where each entry is a tuple of (column, type).
    """
    print(f' - DB CALL: describe_table({table_name})')

    cursor = conn.cursor()
    cursor.execute(f"PRAGMA table_info({table_name});")

    schema = cursor.fetchall()
    # [column index, column name, column type, ...]
    return [(col[1], col[2]) for col in schema]


tbl_name = ["products", "staff", "orders"]
for name in tbl_name:
    table_name = describe_table(name)
    print(table_name)

 - DB CALL: describe_table(products)
[('product_id', 'INTEGER'), ('product_name', 'VARCHAR(255)'), ('price', 'DECIMAL(10, 2)')]
 - DB CALL: describe_table(staff)
[('staff_id', 'INTEGER'), ('first_name', 'VARCHAR(255)'), ('last_name', 'VARCHAR(255)')]
 - DB CALL: describe_table(orders)
[('order_id', 'INTEGER'), ('customer_name', 'VARCHAR(255)'), ('staff_id', 'INTEGER'), ('product_id', 'INTEGER')]


In [8]:
#Query table in a database
def execute_query(sql: str) -> list[list[str]]:
    """Execute an SQL statement, returning the results."""
    print(f' - DB CALL: execute_query({sql})')

    cursor = conn.cursor()
    cursor.execute(sql)
    return cursor.fetchall()

execute_query("SELECT * from products;")

 - DB CALL: execute_query(SELECT * from products;)


[(1, 'Laptop', 799.99), (2, 'Keyboard', 129.99), (3, 'Mouse', 29.99)]

In [9]:
# These are the Python functions defined above.
db_tools = [table_list, describe_table, execute_query]

instruction = """You are a helpful chatbot that can interact with an SQL database
for a computer store. You will take the users questions and turn them into SQL
queries using the tools available. Once you have the information you need, you will
answer the user's question using the data returned.

Use table_list to see what tables are present, describe_table to understand the
schema, and execute_query to issue an SQL SELECT query."""

client = genai.Client(api_key=googl_api_key)

# Start a chat with automatic function calling enabled.
chat = client.chats.create(
    model="gemini-2.0-flash",
    config=types.GenerateContentConfig(
        system_instruction=instruction,
        tools=db_tools,
    ),
)

In [10]:
#Engaging with genai to analyse data in db
resp = chat.send_message("What is the cheapest product")
print(f"\n{resp.text}")

 - DB CALL: execute_query(SELECT ProductName, Price FROM Products ORDER BY Price ASC LIMIT 1;)
 - DB CALL: describe_table(Products)
 - DB CALL: execute_query(SELECT product_name, price FROM Products ORDER BY price ASC LIMIT 1;)

The cheapest product is Mouse which costs 29.99.


In [11]:
chat = client.chats.create(
    model="gemini-2.0-flash",
    config=types.GenerateContentConfig(
        system_instruction=instruction,
        tools=db_tools,
    ),
)

response = chat.send_message("What is the cheapest product")
print(f"\n{response.text}")

 - DB CALL: execute_query(SELECT product_name, price FROM Products ORDER BY price ASC LIMIT 1)

The cheapest product is the Mouse, which costs $29.99.


In [12]:
chat = client.chats.create(
    model="gemini-2.0-flash",
    config=types.GenerateContentConfig(
        system_instruction=instruction,
        tools=db_tools,
    ),
)

response = chat.send_message('What products should salesperson Alice focus on to round out her portfolio? Explain why.')
print(f"\n{response.text}")

 - DB CALL: describe_table(staff)
 - DB CALL: describe_table(orders)
 - DB CALL: describe_table(products)
 - DB CALL: execute_query(SELECT product_name FROM products WHERE product_id IN (SELECT product_id FROM orders WHERE staff_id = (SELECT staff_id FROM staff WHERE first_name = 'Alice')))
 - DB CALL: execute_query(SELECT product_name FROM products)

Alice sells Laptops and Mouses. The store sells Laptops, Keyboards, and Mouses. Therefore, Alice should focus on selling Keyboards to round out her portfolio.


In [13]:
import textwrap


def print_chat_turns(chat):
    """Prints out each turn in the chat history, including function calls and responses."""
    for event in chat.get_history():
        print(f"{event.role.capitalize()}:")

        for part in event.parts:
            if txt := part.text:
                print(f'  "{txt}"')
            elif fn := part.function_call:
                args = ", ".join(f"{key}={val}" for key, val in fn.args.items())
                print(f"  Function call: {fn.name}({args})")
            elif resp := part.function_response:
                print("  Function response:")
                print(textwrap.indent(str(resp.response['result']), "    "))

        print()


print_chat_turns(chat)

User:
  "What products should salesperson Alice focus on to round out her portfolio? Explain why."

Model:
  Function call: table_list()

User:
  Function response:
    ['productions', 'sqlite_sequence', 'staff', 'orders', 'products']

Model:
  "Okay, the tables are: productions, sqlite_sequence, staff, orders, and products.

To figure out what Alice should focus on, I need to see her sales history and compare it to the overall product sales. I'll start by describing the `staff`, `orders`, and `products` tables.
"
  Function call: describe_table(table_name=staff)
  Function call: describe_table(table_name=orders)
  Function call: describe_table(table_name=products)

User:
  Function response:
    [('staff_id', 'INTEGER'), ('first_name', 'VARCHAR(255)'), ('last_name', 'VARCHAR(255)')]
  Function response:
    [('order_id', 'INTEGER'), ('customer_name', 'VARCHAR(255)'), ('staff_id', 'INTEGER'), ('product_id', 'INTEGER')]
  Function response:
    [('product_id', 'INTEGER'), ('product_name

In [14]:
from pprint import pformat
from IPython.display import display, Image, Markdown

async def handle_response(stream, tool_impl=None):
    """Stream output and handle any tool calls during the session"""

    all_response = []
    async for msg in stream.receive():
        all_response.append(msg)

        if text := msg.text:
            #Ouput any chunk that are stream
            if len(all_response) < 2 or not all_response[-2].text:
                display(Markdown(" ### Text"))
            print(text, end="")
        elif tool_call := msg.tool_call:
            #Handle tool call requests
            for fc in tool_call.function_calls:
                display(Markdown(" ### Tool Call"))

                #Execute the tool and collect the result to return to the model
                if callable(tool_impl):
                    try:
                        result = tool_impl(**fc.args)
                    except Exception as e:
                        result = str(e)
                else:
                    result = "ok"

                tool_response = types.LiveClientToolResponse(
                    function_responses=[types.FunctionResponse(
                        name=fc.name,
                        id=fc.id,
                        response={"result": result},
                    )]
                )
                await stream.send(input=tool_response)
        elif msg.server_content and msg.server_content.model_turn:
            #Print any messages showing code the model generate and ran
            for part in msg.server_content.model_turn.parts:
                if code := part.executable_code:
                    display(Markdown(f"### Code\n'''\n{code.code}\n''''"))

                elif result := part.code_execution_result:
                    display(Markdown(f'### Result: {result.outcome}\n'
                             f'```\n{pformat(result.output)}\n```'))

                elif img := part.inline_data:
                    display(Image(img.data))
    print()
    return all_response


In [15]:
model = 'gemini-2.0-flash-exp'
live_client = genai.Client(api_key=googl_api_key,
                           http_options=types.HttpOptions(api_version='v1alpha'))

# Wrap the existing execute_query tool you used in the earlier example.
execute_query_tool_def = types.FunctionDeclaration.from_callable(
    client=live_client, callable=execute_query)

# Provide the model with enough information to use the tool, such as describing
# the database so it understands which SQL syntax to use.
sys_int = """You are a database interface. Use the `execute_query` function
to answer the users questions by looking up information in the database,
running any necessary queries and responding to the user.

You need to look up table schema using sqlite3 syntax SQL, then once an
answer is found be sure to tell the user. If the user is requesting an
action, you must also execute the actions.
"""

config = {
    "response_modalities": ["TEXT"],
    "system_instruction": {"parts": [{"text": sys_int}]},
    "tools": [
        {"code_execution": {}},
        {"function_declarations": [execute_query_tool_def.to_json_dict()]},
    ],
}

async with live_client.aio.live.connect(model=model, config=config) as session:

  message = "Please generate and insert 5 new rows in the orders table."
  print(f"> {message}\n")

  await session.send(input=message, end_of_turn=True)
  await handle_response(session, tool_impl=execute_query)

/tmp/ipykernel_36333/3097181799.py:29: ExperimentalWarning: The live API is experimental and may change in future versions.
  async with live_client.aio.live.connect(model=model, config=config) as session:


> Please generate and insert 5 new rows in the orders table.



 ### Text

I need to understand the structure of the `orders` table before I can insert new rows. I'll start by querying the database schema.


### Code
'''
print(default_api.execute_query(sql="PRAGMA table_info(orders);"))

''''

...

 ### Tool Call

 - DB CALL: execute_query(INSERT INTO orders (customer_name, staff_id, product_id) VALUES ('Eve Miller', 2, 105);)


### Result: Outcome.OUTCOME_OK
```
("{'result': []}\n"
 "{'result': []}\n"
 "{'result': []}\n"
 "{'result': []}\n"
 "{'result': []}\n")
```

 ### Text

I have successfully inserted 5 new rows into the `orders` table.



In [16]:
#Plotting the db
async with live_client.aio.live.connect(model=model, config=config) as session:

  message = "Can you figure out the number of orders that were made by each of the staff?"
  print(f"> {message}\n")

  await session.send(input=message, end_of_turn=True)
  await handle_response(session, tool_impl=execute_query)

> Can you figure out the number of orders that were made by each of the staff?



### Code
'''
sql_query = "SELECT name FROM sqlite_master WHERE type='table';"
print(default_api.execute_query(sql=sql_query))

''''

 ### Tool Call

 - DB CALL: execute_query(SELECT name FROM sqlite_master WHERE type='table';)


### Result: Outcome.OUTCOME_OK
```
("{'result': [['productions'], ['sqlite_sequence'], ['staff'], ['orders'], "
 "['products']]}\n")
```

...

 ### Tool Call

 - DB CALL: execute_query(
SELECT s.first_name, s.last_name, COUNT(o.order_id) AS order_count
FROM staff s
LEFT JOIN orders o ON s.staff_id = o.staff_id
GROUP BY s.staff_id
)


### Result: Outcome.OUTCOME_OK
```
("{'result': [['Alice', 'Smith', 4], ['Bob', 'Johnson', 3], ['Charlie', "
 "'Williams', 1]]}\n")
```

 ### Text

Okay, the query results show the number of orders made by each staff member. Alice Smith has 4 orders, Bob Johnson has 3 orders and Charlie Williams has 1 order.



In [22]:
async with live_client.aio.live.connect(model=model, config=config) as session:

  message = "how many products where ordered by the staff"
  print(f"> {message}\n")

  await session.send(input=message, end_of_turn=True)
  await handle_response(session, tool_impl=execute_query)

> how many products where ordered by the staff



### Code
'''
sql_query = "SELECT COUNT(*) FROM products WHERE ordered_by = 'staff';"
result = default_api.execute_query(sql=sql_query)
print(result)

''''

 ### Tool Call

 - DB CALL: execute_query(SELECT COUNT(*) FROM products WHERE ordered_by = 'staff';)


### Result: Outcome.OUTCOME_OK
```
"{'result': 'no such column: ordered_by'}\n"
```

 ### Text

I am sorry, it seems that there is no column named 'ordered_by' in the products table. To assist you better, could you provide the correct column name that indicates who ordered the product? Alternatively, could you describe which columns are available in the products table so I can adjust my query?

